In [1]:
# 데이터 및 모델, 라이브러리 불러오기
import pandas as pd
import joblib
from scipy.sparse import csr_matrix, hstack

# 팀2 결과(list[dict])를 DataFrame으로 변환
# 팀2 결과가 없으면 기존 CSV를 사용하여 테스트
def load_priority_input(team2_output=None):

    # 실제 연동 시
    if team2_output is not None:
        if not team2_output:
            raise ValueError("팀2 결과가 비어 있습니다.")

        return pd.DataFrame(team2_output)

    # 테스트용
    return pd.read_csv("cve_preprocessed.csv")


# -------------------------------
# 테스트 시
df = load_priority_input()

# 실제 연동 시
# team2_output 변수에 팀2 결과를 넣고 아래 줄 주석 해제
# 위의 df = load_priority_input() 줄은 주석 처리
# df = load_priority_input(team2_output)
# -------------------------------

# 학습된 모델 불러오기
model = joblib.load("risk_model.pkl")

print("입력 데이터 크기 :", df.shape)

입력 데이터 크기 : (9994, 9)


In [2]:
# 팀2 입력 데이터의 필수 컬럼 확인
required_columns = [
    "cve_id",
    "cvss_score",
    "attack_vector",
    "attack_complexity",
    "privileges_required",
    "user_interaction",
    "cwe",
    "description",
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"필수 컬럼이 없습니다: {missing_columns}"
    )

print("필수 컬럼 확인 완료")

필수 컬럼 확인 완료


In [3]:
# 문자열 및 결측치 정리
categorical_columns = [
    "attack_vector",
    "attack_complexity",
    "privileges_required",
    "user_interaction",
    "cwe",
]

for column in categorical_columns:
    df[column] = (
        df[column]
        .fillna("UNKNOWN")
        .astype(str)
        .str.strip()
        .str.upper()
    )

df["description"] = (
    df["description"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df["cvss_score"] = pd.to_numeric(
    df["cvss_score"],
    errors="coerce",
).fillna(0.0)

print("입력 데이터 정리 완료")

입력 데이터 정리 완료


In [4]:
#저장한 Feature 객체 불러오기
label_encoder = joblib.load("feature_data.pkl")["label_encoder"]
tfidf_vectorizer = joblib.load("tfidf_vectorizer.pkl")
feature_cols = joblib.load("feature_cols.pkl")
encoded_columns = joblib.load("encoded_columns.pkl")

# 1. 범주형 원-핫 인코딩 (학습 때와 동일 방식)
encoded_features = pd.get_dummies(
    df[feature_cols],
    prefix=feature_cols,
    dtype=int,
)
encoded_features = encoded_features.reindex(columns=encoded_columns, fill_value=0)

# 2. description TF-IDF 변환 (transform만! fit 하면 안 됨)
description_features = tfidf_vectorizer.transform(df["description"])

# 3. 두 결과 결합 (학습 때와 같은 순서)
categorical_features = csr_matrix(encoded_features.values)
X_new = hstack([categorical_features, description_features], format="csr")

# 4. 예측 + 라벨 복원
df["predicted_severity"] = label_encoder.inverse_transform(model.predict(X_new))

print("예측 완료")
print(df[["cve_id", "predicted_severity"]].head())

예측 완료
           cve_id predicted_severity
0  CVE-2026-57752               HIGH
1  CVE-2026-57753               HIGH
2  CVE-2026-57754             MEDIUM
3  CVE-2026-57755             MEDIUM
4  CVE-2026-57756               HIGH


In [5]:
#점수표
severity_score_map = {
    "LOW": 25,
    "MEDIUM": 50,
    "HIGH": 75,
    "CRITICAL": 100,
}

attack_vector_score_map = {
    "PHYSICAL": 20,
    "LOCAL": 40,
    "ADJACENT_NETWORK": 70,
    "NETWORK": 100,
}

attack_complexity_score_map = {
    "HIGH": 40,
    "LOW": 100,
}

privileges_score_map = {
    "HIGH": 30,
    "LOW": 70,
    "NONE": 100,
}

user_interaction_score_map = {
    "REQUIRED": 40,
    "NONE": 100,
}

In [6]:
#우선순위 함수
def calculate_priority_score(row) -> float:
    cvss_score = float(row["cvss_score"]) * 10

    severity_score = severity_score_map.get(
        row["predicted_severity"],
        0,
    )

    vector_score = attack_vector_score_map.get(
        row["attack_vector"],
        0,
    )

    complexity_score = attack_complexity_score_map.get(
        row["attack_complexity"],
        0,
    )

    privilege_score = privileges_score_map.get(
        row["privileges_required"],
        0,
    )

    interaction_score = user_interaction_score_map.get(
        row["user_interaction"],
        0,
    )

    score = (
        cvss_score * 0.50
        + severity_score * 0.20
        + vector_score * 0.10
        + complexity_score * 0.08
        + privilege_score * 0.07
        + interaction_score * 0.05
    )

    return round(score, 2)

In [7]:
# 우선순위 점수를 대응 등급으로 변환
def classify_response_priority(
    score: float,
) -> str:
    if score >= 85:
        return "긴급 대응"

    if score >= 70:
        return "우선 대응"

    if score >= 50:
        return "검토 필요"

    return "일반 관리"

In [8]:
# 우선순위 점수 생성
df["priority_score"] = df.apply(
    calculate_priority_score,
    axis=1,
)

# 대응 등급 생성
df["response_priority"] = (
    df["priority_score"].apply(
        classify_response_priority
    )
)

# 우선순위 점수 기준 정렬
df = df.sort_values(
    by=[
        "priority_score",
        "cvss_score",
    ],
    ascending=False,
).reset_index(drop=True)

# 우선순위 순번 생성
df["priority_rank"] = df.index + 1

display(
    df[
        [
            "priority_rank",
            "cve_id",
            "cvss_score",
            "predicted_severity",
            "attack_vector",
            "attack_complexity",
            "privileges_required",
            "user_interaction",
            "priority_score",
            "response_priority",
        ]
    ].head(20)
)

# 우선순위 점수 분포 확인
print(
    df[
        "priority_score"
    ].value_counts().head(10)
)

,priority_rank,cve_id,cvss_score,predicted_severity,attack_vector,attack_complexity,privileges_required,user_interaction,priority_score,response_priority
0,1,CVE-2026-50746,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0,긴급 대응
1,2,CVE-2026-56004,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0,긴급 대응
2,3,CVE-2026-13768,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0,긴급 대응
3,4,CVE-2026-48316,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0,긴급 대응
4,5,CVE-2026-54763,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0,긴급 대응
5,6,CVE-2026-57572,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0,긴급 대응
6,7,CVE-2026-59726,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0,긴급 대응
7,8,CVE-2026-54769,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0,긴급 대응
8,9,CVE-2026-61447,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0,긴급 대응
9,10,CVE-2026-54917,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0,긴급 대응


priority_score
82.5    796
59.4    734
99.0    425
86.0    406
69.5    362
75.9    333
70.4    327
58.5    324
77.5    278
86.9    274
Name: count, dtype: int64


In [9]:
# 실제 Severity 컬럼이 있을 경우 분포 출력
if "severity" in df.columns:
    print("실제 정답 분포")
    print(
        df[
            "severity"
        ].value_counts()
    )

print("\n모델 예측 분포")
print(
    df[
        "predicted_severity"
    ].value_counts()
)

실제 정답 분포
severity
HIGH        4352
MEDIUM      4101
CRITICAL    1194
LOW          347
Name: count, dtype: int64

모델 예측 분포
predicted_severity
MEDIUM      4293
HIGH        4086
CRITICAL    1238
LOW          377
Name: count, dtype: int64


In [10]:
# 팀4 전달용 컬럼 구성
output_columns = [
    "priority_rank",
    "cve_id",
    "cvss_score",
    "severity",
    "predicted_severity",
    "priority_score",
    "response_priority",
    "attack_vector",
    "attack_complexity",
    "privileges_required",
    "user_interaction",
    "cwe",
    "description",
]

# service, version이 있을 경우 추가
optional_columns = [
    "service",
    "version",
]

for column in optional_columns:
    if column in df.columns:
        output_columns.insert(
            2,
            column,
        )

# 실제 존재하는 컬럼만 사용
output_columns = [
    column
    for column in output_columns
    if column in df.columns
]

# 팀4 전달용 list[dict] 생성
priority_output = (
    df[
        output_columns
    ].to_dict(
        orient="records"
    )
)

print("팀4 전달용 Output 생성 완료")
print("결과 개수:", len(priority_output))
print(priority_output[:2])

팀4 전달용 Output 생성 완료
결과 개수: 9994
[{'priority_rank': 1, 'cve_id': 'CVE-2026-50746', 'cvss_score': 10.0, 'severity': 'CRITICAL', 'predicted_severity': 'CRITICAL', 'priority_score': 100.0, 'response_priority': '긴급 대응', 'attack_vector': 'NETWORK', 'attack_complexity': 'LOW', 'privileges_required': 'NONE', 'user_interaction': 'NONE', 'cwe': 'CWE-284', 'description': 'A malicious actor with access to the network could exploit an Improper Access Control vulnerability found in UniFi Connect Application to execute a Command Injection on the host device.'}, {'priority_rank': 2, 'cve_id': 'CVE-2026-56004', 'cvss_score': 10.0, 'severity': 'CRITICAL', 'predicted_severity': 'CRITICAL', 'priority_score': 100.0, 'response_priority': '긴급 대응', 'attack_vector': 'NETWORK', 'attack_complexity': 'LOW', 'privileges_required': 'NONE', 'user_interaction': 'NONE', 'cwe': 'CWE-78', 'description': 'A shellcode injection in the mercurial handler of the obs tar_scm source service before version 0.12.4 could be used 

In [11]:
# CSV 및 JSON 파일 저장
import json

pd.DataFrame(
    priority_output
).to_csv(
    "priority.csv",
    index=False,
    encoding="utf-8-sig",
)

with open(
    "priority.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        priority_output,
        file,
        ensure_ascii=False,
        indent=4,
    )

print("priority.csv 생성 완료")
print("priority.json 생성 완료")
print("결과 크기:", len(priority_output))

priority.csv 생성 완료
priority.json 생성 완료
결과 크기: 9994
